> 在线讲义：[第 41 章 · Scaling law：幂律与 log-log 拟合、Kaplan 与 Chinchilla、计算最优、6ND、外推误差](https://wikieden.github.io/math-and-eval-stats/lessons/41-scaling-law.html) · 本书《数学与实验统计》

# 第 41 章 · Scaling law：幂律与 log-log 拟合、Kaplan 与 Chinchilla、计算最优、6ND、外推误差

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/41-scaling-law.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 拟合 | log-log 直线 | `fit_power(c, l, 2.0)` | b = 0.0929 |
| 第二步 外推 | a·C^{−b} | `extrapolate(a, b, 1e22, 2.0)` | 2.2105（真值 2.1893） |
| 第三步 Chinchilla | E + A/N^α + B/D^β | `chinchilla_loss(70e9, 1.4e12)` | 1.9366 |
| 第三步 最优指数 | β/(α+β) | `opt_exponents()` | 0.452、0.548 |
| 第四步 6ND | 6·N·D | `flops_6nd(1e9, 20e9)` | 1.2e20 |

例子：用一个已知的幂律 L(C) = 2.0 + 30·C^{−0.1} 造“实验点”（加 1% 噪声），只用小算力的点拟合，再往大算力外推；
Chinchilla 参数化形式 L(N, D) = E + A/N^α + B/D^β 的系数取论文 Approach 3 的拟合值；6ND 计算账。
里程碑 ME-M26 的“用 Chinchilla 表里的点做拟合、给定 FLOPs 求最优 N / D、估 7B 的 GPU 小时”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
E, A, B, ALPHA, BETA = 1.69, 406.4, 410.7, 0.34, 0.28   # Chinchilla Approach 3（Hoffmann 2022 式 (10)）

**函数 `true_loss`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def true_loss(c):
    return 2.0 + 30.0 * np.asarray(c, float) ** -0.1

**函数 `make_points`**：算力 1e15 到 1e18 之间取 n 个点，loss 加 1% 的乘性噪声。

In [ ]:
def make_points(seed: int = 0, n: int = 8) -> tuple[np.ndarray, np.ndarray]:
    """算力 1e15 到 1e18 之间取 n 个点，loss 加 1% 的乘性噪声。"""
    rng = np.random.default_rng(seed)
    c = np.logspace(15, 18, n)
    return c, true_loss(c) * (1 + 0.01 * rng.standard_normal(n))

**函数 `fit_power`**：在 log-log 上拟合 L − E0 = a·C^{−b}（E0 是假定的不可约 loss）。返回 (a, b)。

In [ ]:
def fit_power(c, l, irreducible: float = 0.0) -> tuple[float, float]:
    """在 log-log 上拟合 L − E0 = a·C^{−b}（E0 是假定的不可约 loss）。返回 (a, b)。"""
    slope, inter = np.polyfit(np.log(c), np.log(np.asarray(l) - irreducible), 1)
    return float(np.exp(inter)), float(-slope)

**函数 `extrapolate`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def extrapolate(a: float, b: float, c, irreducible: float = 0.0):
    return irreducible + a * np.asarray(c, float) ** -b

**函数 `chinchilla_loss`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def chinchilla_loss(n, d):
    return E + A / np.asarray(n, float) ** ALPHA + B / np.asarray(d, float) ** BETA

**函数 `opt_exponents`**：C = 6ND 固定时最小化 L(N, D)：N_opt ∝ C^{β/(α+β)}，D_opt ∝ C^{α/(α+β)}。

In [ ]:
def opt_exponents() -> tuple[float, float]:
    """C = 6ND 固定时最小化 L(N, D)：N_opt ∝ C^{β/(α+β)}，D_opt ∝ C^{α/(α+β)}。"""
    return BETA / (ALPHA + BETA), ALPHA / (ALPHA + BETA)

**函数 `flops_6nd`**：训练 FLOPs ≈ 6·N·D（前向 2ND、反向 4ND）。

In [ ]:
def flops_6nd(n: float, d: float) -> float:
    """训练 FLOPs ≈ 6·N·D（前向 2ND、反向 4ND）。"""
    return 6 * n * d

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 幂律在 log-log 上是直线")
c, l = make_points()
a, b = fit_power(c, l, irreducible=2.0)
print(f"  已知不可约 loss 2.0：拟合 a = {a:.2f}、b = {b:.4f}（真值 30、0.1）")
a0, b0 = fit_power(c, l, irreducible=0.0)
print(f"  忘了减不可约项：拟合指数 b = {b0:.4f}（被严重低估）")

In [ ]:
print("\n第二步 外推：差一点的指数，放大到 1000 倍算力")
for cc in (1e18, 1e20, 1e22):
    print(f"  C = {cc:.0e}：真值 {true_loss(cc):.4f}，正确拟合 {extrapolate(a, b, cc, 2.0):.4f}，忘减常数 {extrapolate(a0, b0, cc):.4f}")

In [ ]:
print("\n第三步 Chinchilla 的参数化形式 L(N, D) = E + A/N^α + B/D^β")
for n, d in ((70e9, 1.4e12), (280e9, 300e9)):
    print(f"  N = {n / 1e9:.0f}B、D = {d / 1e9:.0f}B：L = {chinchilla_loss(n, d):.4f}，FLOPs 6ND = {flops_6nd(n, d):.2e}")
an, ad = opt_exponents()
print(f"  算力固定时 N_opt ∝ C^{an:.3f}，D_opt ∝ C^{ad:.3f}（Kaplan 2020 给的是 N ∝ C^0.73）")

In [ ]:
print("\n第四步 6ND 计算账")
for n, d in ((1e9, 20e9), (1e9, 1e12)):
    print(f"  N = {n / 1e9:.0f}B、D = {d / 1e9:.0f}B：6ND = {flops_6nd(n, d):.2e} FLOPs，D/N = {d / n:.0f}")

## 3. 代码里才看得到的

1. **噪声是乘性的**：loss 乘以 (1 + 0.01·ε)，在对数空间里近似加性，正好符合 log-log 拟合的假设。
2. **拟合时假设 E 已知**：真实拟合要把 E 也当参数，用 `curve_fit` 或 Huber 损失；这里固定 E 是为了把“减不减常数”的影响单独拿出来看。
3. **没有用 Chinchilla 论文表里的数据点**：拟合真实数据、按给定 FLOPs 求最优 N / D、估 7B 的 GPU 小时，留给自己做（里程碑 ME-M26）。

## 4. 自己改一改

1. 把噪声从 1% 改成 3%，指数和外推误差怎么变？
2. 把拟合点的范围改成 10¹⁵–10²⁰，外推到 10²² 的误差能小多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 幂律先减底，log-log 才是直线；
> 外推放大的是指数误差，形式错了更离谱；
> Chinchilla 参数数据同步长，约二十 token 一个参数；
> 训练算力六 N D；推理多就故意过训练。